<a href="https://colab.research.google.com/github/khing-fm/Data_Science/blob/main/EHU_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
EHU Dashboard Data - EDA, Visualization, Machine Learning and Prediction
=========================================================================

Analyses the Environmental Health Unit (EHU) inspection workbook:

    1. Load & clean      - read all sheets, fix types, engineer features
    2. EDA               - summaries, compliance patterns, documents, nuisances
    3. Visualization     - saved as PNG files in <output_dir>/figures
    4. Classification    - predict inspection non-compliance (leakage-aware)
    5. Regression        - predict number of nuisances found at a premises
    6. Forecasting       - weekly inspection volume / non-compliance outlook
    7. Prediction        - risk scoring of new inspections + exported tables

Usage
-----
    python ehu_analysis.py
    python ehu_analysis.py --data EHU_Dashboard_Data.xlsx --out ehu_output

Requirements: pandas, numpy, matplotlib, seaborn, scikit-learn, openpyxl
"""

from __future__ import annotations

import argparse
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe; figures are saved, not shown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    f1_score,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

In [4]:
# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
DOC_FLAGS = [
    "HasFumigationCert",
    "HasWaterAnalysis",
    "HasFoodHandlerTest",
    "HasWasteDisposalEvidence",
]
CATEGORICAL_PLANNING = ["Division", "PremisesType", "PremisesClassGrp", "InspectionType"]
NUMERIC_PLANNING = ["MonthNum", "DayOfWeek", "Latitude", "Longitude"]
# On-site observations: available only once the inspector is at the premises.
NUMERIC_ONSITE = DOC_FLAGS + ["HasHACCP", "CoreDocsSighted"]

# Columns deliberately NOT used as predictors of compliance (they are outcomes
# of the inspection itself and would leak the answer):
#   Findings, IsCompliant, AbatementNoticeServed, AbatementServedFlag,
#   NuisanceCount, RemarkCategory, MissingDocsNote, NoDocumentSighted


@dataclass
class Paths:
    data: Path
    out: Path

    @property
    def fig(self) -> Path:
        return self.out / "figures"

    def make(self) -> None:
        self.fig.mkdir(parents=True, exist_ok=True)


def banner(title: str) -> None:
    print(f"\n{'=' * 78}\n{title}\n{'=' * 78}")


def save_fig(fig: plt.Figure, paths: Paths, name: str) -> None:
    fig.tight_layout()
    fig.savefig(paths.fig / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved figures/{name}.png")

In [5]:
# --------------------------------------------------------------------------- #
# 1. Load & clean
# --------------------------------------------------------------------------- #
def load_data(path: Path) -> dict[str, pd.DataFrame]:
    sheets = pd.read_excel(path, sheet_name=None)
    banner("1. DATA LOADED")
    for name, df in sheets.items():
        print(f"  {name:<18} {df.shape[0]:>5} rows x {df.shape[1]:>2} cols")
    return sheets


def clean_inspections(sheets: dict[str, pd.DataFrame], top_classes: int = 8) -> pd.DataFrame:
    """Return a modelling-ready inspections table."""
    df = sheets["Fact_Inspections"].copy()
    centroids = sheets["Dim_Division"].set_index("Division")

    df["InspectionDate"] = pd.to_datetime(df["InspectionDate"])
    df = df.sort_values("InspectionDate").reset_index(drop=True)

    # Missing coordinates -> division centroid
    for col, cen in [("Latitude", "CentroidLat"), ("Longitude", "CentroidLon")]:
        df[col] = df[col].fillna(df["Division"].map(centroids[cen]))

    # Text clean-up
    df["OrganizationName"] = df["OrganizationName"].fillna("Unknown")
    df["MissingDocsNote"] = df["MissingDocsNote"].fillna("")
    df["PremisesDetail"] = df["PremisesDetail"].fillna("Unspecified")

    # Time features
    df["DayOfWeek"] = df["InspectionDate"].dt.dayofweek
    df["WeekStart"] = df["InspectionDate"].dt.to_period("W-SUN").dt.start_time
    df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)

    # Collapse the long tail of premises classes (27 -> top N + "Other")
    top = df["PremisesClass"].value_counts().head(top_classes).index
    df["PremisesClassGrp"] = df["PremisesClass"].where(df["PremisesClass"].isin(top), "Other")

    # Compact inspector grouping (for EDA only)
    def inspector_group(code: str) -> str:
        code = str(code)
        if code.startswith("Officer"):
            return "Numbered officer"
        if code.startswith("IKJ"):
            return "IKJ team"
        return "Generic / not recorded"

    df["InspectorGroup"] = df["InspectorCode"].map(inspector_group)
    df["MissingCoreDocs"] = 4 - df["CoreDocsSighted"]
    return df

In [6]:
# --------------------------------------------------------------------------- #
# 2 & 3. EDA + visualization
# --------------------------------------------------------------------------- #
def run_eda(df: pd.DataFrame, sheets: dict[str, pd.DataFrame], paths: Paths) -> None:
    banner("2. EXPLORATORY DATA ANALYSIS")

    print(f"Period            : {df.InspectionDate.min():%d %b %Y} -> {df.InspectionDate.max():%d %b %Y}")
    print(f"Inspections       : {len(df):,}   (unique IDs: {df.InspectionID.is_unique})")
    print(f"Non-compliance    : {df.IsNonCompliant.mean():.1%}")
    print(f"Fully documented  : {df.FullyDocumented.mean():.1%}")
    print(f"Mean nuisances    : {df.NuisanceCount.mean():.2f} per inspection")

    print("\nMissing values (columns with any):")
    miss = sheets["Fact_Inspections"].isna().sum()
    print(miss[miss > 0].to_string())

    print("\nNon-compliance rate by Division:")
    div = df.groupby("Division").agg(
        inspections=("InspectionID", "count"),
        noncompliant_rate=("IsNonCompliant", "mean"),
        avg_nuisances=("NuisanceCount", "mean"),
    )
    print(div.round(3).sort_values("noncompliant_rate", ascending=False).to_string())

    print("\nNon-compliance rate by Premises Type:")
    ptype = df.groupby("PremisesType").agg(
        inspections=("InspectionID", "count"), noncompliant_rate=("IsNonCompliant", "mean")
    )
    print(ptype.round(3).to_string())

    print("\nDocument availability (share of inspections where sighted):")
    docs = df[DOC_FLAGS + ["HasHACCP"]].mean().sort_values()
    print(docs.round(3).to_string())

    nuis = sheets["Fact_Nuisances"].merge(
        df[["InspectionID", "Division", "PremisesType"]], on="InspectionID", how="left"
    )
    print("\nTop nuisance themes:")
    print(nuis["NuisanceTheme"].value_counts().head(8).to_string())

    # ---- Figures ---------------------------------------------------------- #
    print("\nBuilding figures...")

    # 1. Monthly volume + compliance rate
    monthly = df.groupby(df.InspectionDate.dt.to_period("M")).agg(
        inspections=("InspectionID", "count"), nc_rate=("IsNonCompliant", "mean")
    )
    monthly.index = monthly.index.to_timestamp()
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(monthly.index, monthly["inspections"], width=20, color="#4C78A8", label="Inspections")
    ax.set_ylabel("Inspections")
    ax2 = ax.twinx()
    ax2.plot(monthly.index, monthly["nc_rate"] * 100, color="#E45756", marker="o", label="Non-compliance %")
    ax2.set_ylabel("Non-compliance rate (%)")
    ax2.set_ylim(0, 100)
    ax2.grid(False)
    ax.set_title("Monthly inspection volume and non-compliance rate")
    save_fig(fig, paths, "01_monthly_trend")

    # 2. Division & premises type
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    div["noncompliant_rate"].sort_values().mul(100).plot.barh(ax=axes[0], color="#E45756")
    axes[0].set_title("Non-compliance rate by division (%)")
    axes[0].set_xlabel("%")
    ptype["noncompliant_rate"].sort_values().mul(100).plot.barh(ax=axes[1], color="#F58518")
    axes[1].set_title("Non-compliance rate by premises type (%)")
    axes[1].set_xlabel("%")
    save_fig(fig, paths, "02_noncompliance_by_division_type")

    # 3. Top premises classes
    cls = (
        df.groupby("PremisesClass")
        .agg(n=("InspectionID", "count"), nc=("IsNonCompliant", "mean"))
        .query("n >= 15")
        .sort_values("nc")
    )
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(cls.index, cls["nc"] * 100, color="#54A24B")
    for i, (n, nc) in enumerate(zip(cls["n"], cls["nc"])):
        ax.text(nc * 100 + 1, i, f"n={n}", va="center", fontsize=8)
    ax.set_title("Non-compliance rate by premises class (classes with n >= 15)")
    ax.set_xlabel("%")
    ax.set_xlim(0, 105)
    save_fig(fig, paths, "03_noncompliance_by_class")

    # 4. Document availability
    fig, ax = plt.subplots(figsize=(8, 4))
    (docs * 100).plot.barh(ax=ax, color="#72B7B2")
    ax.set_title("Documents sighted during inspections (%)")
    ax.set_xlabel("% of inspections")
    save_fig(fig, paths, "04_document_availability")

    # 5. Documents vs compliance
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=df, x="CoreDocsSighted", y="IsNonCompliant", color="#B279A2", ax=ax, errorbar=None)
    ax.set_title("Non-compliance rate vs number of core documents sighted (0-4)")
    ax.set_ylabel("Non-compliance rate")
    save_fig(fig, paths, "05_docs_vs_noncompliance")

    # 6. Nuisance themes + theme x division heatmap
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1, 1.2]})
    nuis["NuisanceTheme"].value_counts().sort_values().plot.barh(ax=axes[0], color="#9D755D")
    axes[0].set_title("Nuisance themes recorded")
    heat = pd.crosstab(nuis["NuisanceTheme"], nuis["Division"], normalize="columns") * 100
    sns.heatmap(heat, annot=True, fmt=".0f", cmap="YlOrRd", ax=axes[1], cbar_kws={"label": "% of division's nuisances"})
    axes[1].set_title("Nuisance theme mix by division (%)")
    axes[1].set_ylabel("")
    save_fig(fig, paths, "06_nuisance_themes")

    # 7. Nuisance count distribution (non-compliant only)
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.countplot(data=df[df.IsNonCompliant == 1], x="NuisanceCount", color="#4C78A8", ax=ax)
    ax.set_title("Nuisances per non-compliant inspection")
    save_fig(fig, paths, "07_nuisance_count_dist")

    # 8. Map-style scatter
    fig, ax = plt.subplots(figsize=(8, 7))
    geo = df[(df.Latitude.between(6.3, 6.8)) & (df.Longitude.between(3.0, 3.8))]
    sns.scatterplot(
        data=geo, x="Longitude", y="Latitude", hue="Findings",
        palette={"Compliant": "#54A24B", "Non-Compliant": "#E45756"}, alpha=0.65, s=35, ax=ax,
    )
    ax.set_title("Inspection locations by outcome")
    save_fig(fig, paths, "08_geo_scatter")

    # 9. Weekday / month heatmap of non-compliance
    pivot = df.pivot_table(index="DayOfWeek", columns="MonthNum", values="IsNonCompliant", aggfunc="mean")
    pivot = pivot.reindex(range(7))
    pivot.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.heatmap(pivot * 100, annot=True, fmt=".0f", cmap="RdYlGn_r", ax=ax, cbar_kws={"label": "% non-compliant"})
    ax.set_title("Non-compliance rate by weekday and month")
    ax.set_xlabel("Month number")
    ax.set_ylabel("")
    save_fig(fig, paths, "09_weekday_month_heatmap")

    # 10. Correlation heatmap
    num_cols = DOC_FLAGS + ["HasHACCP", "CoreDocsSighted", "NuisanceCount", "IsNonCompliant", "AbatementServedFlag"]
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
    ax.set_title("Correlation matrix")
    save_fig(fig, paths, "10_correlation")

    # 11. Inspector workload
    insp = df.groupby("InspectorGroup").agg(n=("InspectionID", "count"), nc=("IsNonCompliant", "mean"))
    print("\nInspector group summary:")
    print(insp.round(3).to_string())
    print("  (note: non-compliance differs sharply by division and recording practice;"
          " compare rates cautiously.)")

In [7]:
# --------------------------------------------------------------------------- #
# 4. Classification
# --------------------------------------------------------------------------- #
def make_preprocessor(cat_cols: list[str], num_cols: list[str], scale: bool) -> ColumnTransformer:
    num_pipe = StandardScaler() if scale else "passthrough"
    return ColumnTransformer(
        [
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=5), cat_cols),
            ("num", num_pipe, num_cols),
        ]
    )


def classifier_zoo(cat_cols: list[str], num_cols: list[str]) -> dict[str, Pipeline]:
    return {
        "Logistic Regression": Pipeline(
            [("prep", make_preprocessor(cat_cols, num_cols, True)),
             ("model", LogisticRegression(max_iter=2000, C=1.0))]
        ),
        "Random Forest": Pipeline(
            [("prep", make_preprocessor(cat_cols, num_cols, False)),
             ("model", RandomForestClassifier(n_estimators=400, min_samples_leaf=3,
                                              random_state=RANDOM_STATE, n_jobs=-1))]
        ),
        "Gradient Boosting": Pipeline(
            [("prep", make_preprocessor(cat_cols, num_cols, False)),
             ("model", GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                  max_depth=3, random_state=RANDOM_STATE))]
        ),
    }


def temporal_split(df: pd.DataFrame, test_frac: float = 0.2) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Train on earlier inspections, test on the most recent ones (no look-ahead)."""
    cut = int(len(df) * (1 - test_frac))
    ordered = df.sort_values("InspectionDate")
    return ordered.iloc[:cut], ordered.iloc[cut:]


def evaluate_feature_set(
    name: str, df: pd.DataFrame, cat_cols: list[str], num_cols: list[str], paths: Paths, tag: str
):
    """Cross-validate several models, then report the best on a temporal hold-out."""
    banner(f"4. CLASSIFICATION - {name}")
    features = cat_cols + num_cols
    X, y = df[features], df["IsNonCompliant"]
    zoo = classifier_zoo(cat_cols, num_cols)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for model_name, pipe in zoo.items():
        res = cross_validate(
            pipe, X, y, cv=cv, n_jobs=-1,
            scoring={"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc"},
        )
        rows.append(
            {
                "model": model_name,
                "accuracy": res["test_accuracy"].mean(),
                "f1": res["test_f1"].mean(),
                "roc_auc": res["test_roc_auc"].mean(),
                "roc_auc_std": res["test_roc_auc"].std(),
            }
        )
    cv_table = pd.DataFrame(rows).set_index("model").sort_values("roc_auc", ascending=False)
    baseline = max(y.mean(), 1 - y.mean())
    print(f"5-fold stratified CV (majority-class baseline accuracy = {baseline:.3f}):")
    print(cv_table.round(3).to_string())

    best_name = cv_table.index[0]
    print(f"\nBest by ROC-AUC: {best_name}")

    # Temporal hold-out (more realistic: predict the future from the past)
    train, test = temporal_split(df)
    best = zoo[best_name].fit(train[features], train["IsNonCompliant"])
    proba = best.predict_proba(test[features])[:, 1]
    pred = (proba >= 0.5).astype(int)
    yt = test["IsNonCompliant"]
    print(f"\nTemporal hold-out ({len(train)} train -> {len(test)} most recent test inspections):")
    print(f"  Accuracy : {accuracy_score(yt, pred):.3f}")
    print(f"  Precision: {precision_score(yt, pred):.3f}")
    print(f"  Recall   : {recall_score(yt, pred):.3f}")
    print(f"  F1       : {f1_score(yt, pred):.3f}")
    print(f"  ROC-AUC  : {roc_auc_score(yt, proba):.3f}")

    # Figures: ROC + confusion matrix
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    RocCurveDisplay.from_predictions(yt, proba, ax=axes[0], name=best_name)
    axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
    axes[0].set_title("ROC curve (temporal hold-out)")
    ConfusionMatrixDisplay.from_predictions(
        yt, pred, ax=axes[1], display_labels=["Compliant", "Non-compliant"], cmap="Blues", colorbar=False
    )
    axes[1].set_title("Confusion matrix")
    save_fig(fig, paths, f"{tag}_roc_confusion")

    # Permutation importance on the full-data fit (grouped by original column)
    final = zoo[best_name].fit(X, y)
    imp = permutation_importance(
        final, X, y, n_repeats=15, random_state=RANDOM_STATE, scoring="roc_auc", n_jobs=-1
    )
    imp_df = (
        pd.DataFrame({"feature": features, "importance": imp.importances_mean, "std": imp.importances_std})
        .sort_values("importance")
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(imp_df["feature"], imp_df["importance"], xerr=imp_df["std"], color="#4C78A8")
    ax.set_title(f"Permutation importance (drop in ROC-AUC) - {name}")
    ax.set_xlabel("Mean decrease in ROC-AUC")
    save_fig(fig, paths, f"{tag}_importance")
    print("\nMost influential features:")
    print(imp_df.sort_values("importance", ascending=False).head(6).round(4).to_string(index=False))
    return final, cv_table, imp_df

In [8]:
# --------------------------------------------------------------------------- #
# 5. Regression - nuisance count
# --------------------------------------------------------------------------- #
def run_regression(df: pd.DataFrame, paths: Paths) -> Pipeline:
    banner("5. REGRESSION - expected nuisance count at non-compliant premises")
    sub = df[df.IsNonCompliant == 1].copy()
    cat_cols = CATEGORICAL_PLANNING
    num_cols = NUMERIC_PLANNING + NUMERIC_ONSITE
    X, y = sub[cat_cols + num_cols], sub["NuisanceCount"]
    print(f"Non-compliant inspections: {len(sub)}; target mean={y.mean():.2f}, std={y.std():.2f}")

    models = {
        "Ridge": Pipeline([("prep", make_preprocessor(cat_cols, num_cols, True)), ("model", Ridge(alpha=5.0))]),
        "Random Forest": Pipeline(
            [("prep", make_preprocessor(cat_cols, num_cols, False)),
             ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=4,
                                             random_state=RANDOM_STATE, n_jobs=-1))]
        ),
        "Gradient Boosting": Pipeline(
            [("prep", make_preprocessor(cat_cols, num_cols, False)),
             ("model", GradientBoostingRegressor(n_estimators=150, learning_rate=0.05,
                                                 max_depth=2, random_state=RANDOM_STATE))]
        ),
    }
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for name, pipe in models.items():
        res = cross_validate(
            pipe, X, y, cv=cv, n_jobs=-1,
            scoring={"mae": "neg_mean_absolute_error", "r2": "r2"},
        )
        rows.append({"model": name, "MAE": -res["test_mae"].mean(), "R2": res["test_r2"].mean()})
    table = pd.DataFrame(rows).set_index("model").sort_values("MAE")
    base_mae = mean_absolute_error(y, np.full(len(y), y.mean()))
    print(f"5-fold CV (predict-the-mean baseline MAE = {base_mae:.3f}):")
    print(table.round(3).to_string())

    best_name = table.index[0]
    best = models[best_name]
    train, test = temporal_split(sub)
    best.fit(train[cat_cols + num_cols], train["NuisanceCount"])
    pred = best.predict(test[cat_cols + num_cols])
    print(f"\nBest: {best_name}. Temporal hold-out: MAE={mean_absolute_error(test['NuisanceCount'], pred):.3f}, "
          f"R2={r2_score(test['NuisanceCount'], pred):.3f}")

    fig, ax = plt.subplots(figsize=(6, 5.5))
    jitter = np.random.default_rng(RANDOM_STATE).normal(0, 0.08, len(pred))
    ax.scatter(test["NuisanceCount"] + jitter, pred, alpha=0.6, color="#4C78A8")
    lim = [0, max(test["NuisanceCount"].max(), pred.max()) + 0.5]
    ax.plot(lim, lim, "k--", alpha=0.5)
    ax.set_xlabel("Actual nuisance count")
    ax.set_ylabel("Predicted")
    ax.set_title(f"Nuisance count: predicted vs actual ({best_name})")
    save_fig(fig, paths, "12_regression_actual_vs_pred")

    if table.loc[best_name, "R2"] < 0.15:
        print("  Note: low R2 - nuisance counts are only weakly explained by these features; "
              "treat predictions as rough averages, not point estimates.")
    return best.fit(X, y)

In [9]:
# --------------------------------------------------------------------------- #
# 6. Forecasting
# --------------------------------------------------------------------------- #
def run_forecast(df: pd.DataFrame, paths: Paths, horizon: int = 4) -> pd.DataFrame:
    banner("6. FORECASTING - weekly inspections and non-compliance")
    last_day = df.InspectionDate.max()
    weekly = (
        df.groupby("WeekStart")
        .agg(inspections=("InspectionID", "count"), noncompliant=("IsNonCompliant", "sum"))
    )
    full_idx = pd.date_range(weekly.index.min(), weekly.index.max(), freq="W-MON")
    weekly = weekly.reindex(full_idx, fill_value=0)
    # drop the final week if it is not complete (its Sunday is after the last data day)
    if weekly.index[-1] + pd.Timedelta(days=6) > last_day:
        weekly = weekly.iloc[:-1]
    print(f"{len(weekly)} complete weeks, {weekly.index.min():%d %b} -> {weekly.index.max():%d %b %Y}")

    def f_mean4(hist: np.ndarray, h: int) -> np.ndarray:
        return np.repeat(hist[-4:].mean(), h)

    def f_trend(hist: np.ndarray, h: int) -> np.ndarray:
        window = hist[-12:]
        t = np.arange(len(hist) - len(window), len(hist))
        slope, intercept = np.polyfit(t, window, 1)
        future = np.arange(len(hist), len(hist) + h)
        return np.clip(slope * future + intercept, 0, None)

    def f_expanding(hist: np.ndarray, h: int) -> np.ndarray:
        return np.repeat(hist.mean(), h)

    methods = {"Last-4-week mean": f_mean4, "Linear trend (12w)": f_trend, "Overall mean": f_expanding}

    def backtest(series: np.ndarray, start: int = 16) -> dict[str, float]:
        errs = {m: [] for m in methods}
        for i in range(start, len(series)):
            for m, fn in methods.items():
                errs[m].append(abs(fn(series[:i], 1)[0] - series[i]))
        return {m: float(np.mean(e)) for m, e in errs.items()}

    out = {}
    fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
    future_idx = pd.date_range(weekly.index[-1] + pd.Timedelta(weeks=1), periods=horizon, freq="W-MON")
    for ax, col, color in zip(axes, ["inspections", "noncompliant"], ["#4C78A8", "#E45756"]):
        series = weekly[col].to_numpy(dtype=float)
        scores = backtest(series)
        best_m = min(scores, key=scores.get)
        fc = methods[best_m](series, horizon)
        print(f"\n{col}: rolling one-step-ahead MAE")
        for m, s in sorted(scores.items(), key=lambda kv: kv[1]):
            print(f"  {m:<22} {s:6.2f}")
        print(f"  -> using '{best_m}'; next {horizon} weeks: {np.round(fc, 1).tolist()}")
        out[col] = fc

        # prediction band from backtest residual spread
        resid = [abs(methods[best_m](series[:i], 1)[0] - series[i]) for i in range(16, len(series))]
        band = 1.25 * np.mean(resid)
        ax.plot(weekly.index, series, color=color, marker="o", ms=3, label="Actual")
        ax.plot(future_idx, fc, color="black", marker="s", ls="--", label=f"Forecast ({best_m})")
        ax.fill_between(future_idx, np.clip(fc - band, 0, None), fc + band, color="grey", alpha=0.25)
        ax.set_ylabel(col.capitalize())
        ax.legend(loc="upper left")
    axes[0].set_title("Weekly inspections and non-compliant findings: history and short-term outlook")
    save_fig(fig, paths, "13_weekly_forecast")

    result = pd.DataFrame(
        {"week_starting": future_idx, "forecast_inspections": out["inspections"].round(1),
         "forecast_noncompliant": out["noncompliant"].round(1)}
    )
    result.to_csv(paths.out / "weekly_forecast.csv", index=False)
    print("\n  Note: only ~40 weekly points exist; treat the forecast as a planning "
          "baseline, not a precise prediction.")
    return result

In [10]:
# --------------------------------------------------------------------------- #
# 7. Prediction / risk scoring
# --------------------------------------------------------------------------- #
def run_prediction(df: pd.DataFrame, planning_model: Pipeline, regressor: Pipeline, paths: Paths) -> None:
    banner("7. PREDICTION & RISK SCORING")
    planning_cols = CATEGORICAL_PLANNING + NUMERIC_PLANNING

    # (a) Risk matrix: expected non-compliance by Division x Premises type
    scored = df.copy()
    scored["pred_risk"] = planning_model.predict_proba(scored[planning_cols])[:, 1]
    matrix = scored.pivot_table(index="Division", columns="PremisesType", values="pred_risk", aggfunc="mean")
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.heatmap(matrix * 100, annot=True, fmt=".0f", cmap="RdYlGn_r", vmin=0, vmax=100, ax=ax,
                cbar_kws={"label": "Predicted non-compliance %"})
    ax.set_title("Pre-visit risk matrix (model-predicted non-compliance %)")
    save_fig(fig, paths, "14_risk_matrix")
    print("Predicted non-compliance risk, Division x Premises type (%):")
    print((matrix * 100).round(0).to_string())

    # (b) Priority list: highest-risk premises classes per division
    priority = (
        scored.groupby(["Division", "PremisesClass"])
        .agg(inspections=("InspectionID", "count"), predicted_risk=("pred_risk", "mean"),
             actual_rate=("IsNonCompliant", "mean"))
        .query("inspections >= 5")
        .sort_values("predicted_risk", ascending=False)
        .round(3)
    )
    priority.to_csv(paths.out / "priority_inspection_targets.csv")
    print("\nTop 10 priority targets (Division / premises class):")
    print(priority.head(10).to_string())

    # (c) Per-inspection scored table (use for audit / follow-up planning)
    cols = ["InspectionID", "InspectionDate", "Division", "PremisesType", "PremisesClass",
            "OrganizationName", "Findings", "pred_risk"]
    scored[cols].sort_values("pred_risk", ascending=False).round({"pred_risk": 3}).to_csv(
        paths.out / "scored_inspections.csv", index=False
    )

    # (d) Demonstration: score brand-new, hypothetical inspections
    new = pd.DataFrame(
        [
            {"Division": "Ikeja", "PremisesType": "Regulated Food Premises", "PremisesClassGrp": "Hotels",
             "InspectionType": "Routine", "MonthNum": 11, "DayOfWeek": 2, "Latitude": 6.57, "Longitude": 3.33},
            {"Division": "Ikorodu-Epe", "PremisesType": "Non-Food Premises", "PremisesClassGrp": "Other",
             "InspectionType": "Complaint", "MonthNum": 11, "DayOfWeek": 3, "Latitude": 6.59, "Longitude": 3.38},
            {"Division": "Lagos Island", "PremisesType": "Regulated Food Premises", "PremisesClassGrp": "Hotels",
             "InspectionType": "Follow-up", "MonthNum": 12, "DayOfWeek": 1, "Latitude": 6.44, "Longitude": 3.43},
        ]
    )
    new["P(non-compliant)"] = planning_model.predict_proba(new[planning_cols])[:, 1].round(3)
    print("\nExample predictions for new (hypothetical) inspections:")
    print(new[["Division", "PremisesType", "PremisesClassGrp", "InspectionType", "P(non-compliant)"]]
          .to_string(index=False))
    print("\n  Re-use: call planning_model.predict_proba(DataFrame[planning_cols]) with the same columns.")

In [12]:
# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #
def main(args_list: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description="EHU dashboard data: EDA, ML and prediction")
    parser.add_argument("--data", default="/content/EHU_Dashboard_Data.xlsx", help="Path to the Excel workbook")
    parser.add_argument("--out", default="ehu_output", help="Output directory")

    # Prevent ipykernel arguments (-f /root/.local/share/...) from crashing the parser
    if args_list is not None:
        args = parser.parse_args(args_list)
    else:
        import sys
        # If running interactively inside Colab without system arguments, use defaults
        if len(sys.argv) > 1 and sys.argv[1] == "-f":
            args = parser.parse_args([])
        else:
            args = parser.parse_args()

    paths = Paths(Path(args.data), Path(args.out))
    paths.make()

    sheets = load_data(paths.data)
    df = clean_inspections(sheets)

    run_eda(df, sheets, paths)

    planning_model, cv_plan, _ = evaluate_feature_set(
        "PRE-VISIT RISK MODEL (no on-site information)", df,
        CATEGORICAL_PLANNING, NUMERIC_PLANNING, paths, "11a_planning",
    )
    _, cv_site, _ = evaluate_feature_set(
        "ON-SITE AUDIT MODEL (adds documents sighted)", df,
        CATEGORICAL_PLANNING, NUMERIC_PLANNING + NUMERIC_ONSITE, paths, "11b_onsite",
    )
    print("\nInterpretation: the pre-visit model shows how well non-compliance can be anticipated "
          "before an inspector arrives; the on-site model shows which observed documents drive the "
          "outcome (it is explanatory, not for pre-visit targeting).")

    regressor = run_regression(df, paths)
    run_forecast(df, paths)
    run_prediction(df, planning_model, regressor, paths)

    banner("DONE")
    print(f"Outputs written to: {paths.out.resolve()}")
    for p in sorted(paths.out.rglob("*")):
        if p.is_file():
            print("  ", p.relative_to(paths.out))


if __name__ == "__main__":
    main()


1. DATA LOADED
  Fact_Inspections     812 rows x 32 cols
  Fact_Nuisances      1066 rows x  5 cols
  Fact_Documents      3248 rows x  4 cols
  Dim_Date             365 rows x 10 cols
  Dim_Premises          28 rows x  3 cols
  Dim_Division           4 rows x  3 cols

2. EXPLORATORY DATA ANALYSIS
Period            : 03 Feb 2026 -> 18 Nov 2026
Inspections       : 812   (unique IDs: True)
Non-compliance    : 48.3%
Fully documented  : 54.1%
Mean nuisances    : 1.31 per inspection

Missing values (columns with any):
PremisesDetail      724
OrganizationName      6
Latitude              1
Longitude             1
MissingDocsNote     688

Non-compliance rate by Division:
              inspections  noncompliant_rate  avg_nuisances
Division                                                   
Ikeja                 120              0.867          3.075
Badagry               101              0.545          1.208
Lagos Island          458              0.467          1.146
Ikorodu-Epe           133   